# sum-back-expand-broadcast — faded example 2: Multi-axis Sum Backward — complete the unsqueeze loop

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `sum-back-expand-broadcast`. Running the beacon reports progress on the `Backprop: sum_back via expand_broadcast` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: sum_back via expand_broadcast` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`sum-back-expand-broadcast`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "sum-back-expand-broadcast"
DD_SUBTOPIC = "Backprop: sum_back via expand_broadcast"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

For a sum over multiple axes with `keepdim=False`, every collapsed axis must be re-inserted as a size-1 axis before expanding. Processing the dimensions in ascending order ensures each `unsqueeze(d)` inserts at the correct position, because axes before `d` were already inserted and only shift positions after `d`.

## Faded exercise 2

The `sum_back_multi` function handles `dims` as a tuple. The `keepdim=True` branch is already complete. **Your task: fill in the `keepdim=False` branch** — loop over `sorted(dims)` and unsqueeze each axis back into `g`, then expand to `x.shape`.

```python
def sum_back_multi(grad_out, x, dims, keepdim=False):
    if keepdim:
        return grad_out.expand(x.shape)
    g = grad_out
    # TODO: for each d in sorted(dims), unsqueeze g along d
    raise NotImplementedError()  # TODO: loop unsqueeze then expand
```

**Fill in:** Iterate over `sorted(dims)`, unsqueeze `g` along each `d`, then return `g.expand(x.shape)`.

In [ ]:
import torch as t

def sum_back_multi(grad_out: t.Tensor, x: t.Tensor, dims: tuple, keepdim: bool = False) -> t.Tensor:
    if keepdim:
        return grad_out.expand(x.shape)
    g = grad_out
    for d in sorted(dims):
        g = g.unsqueeze(d)
    return g.expand(x.shape)

# Quick test
t.manual_seed(5)
x = t.randn(2, 3, 4)
out = x.sum(dim=(0, 2))           # shape (3,)
grad_out = t.ones_like(out)
result = sum_back_multi(grad_out, x, dims=(0, 2), keepdim=False)
print('result.shape:', result.shape)  # expect (2, 3, 4)


def _test():
    import torch as t
    x = t.randn(2, 3, 4)
    # Two-axis sum, keepdim=False
    out = x.sum(dim=(0, 2))
    grad_out = t.ones_like(out)
    result = sum_back_multi(grad_out, x, dims=(0, 2), keepdim=False)
    # Ground truth via autograd
    xr = x.detach().clone().requires_grad_(True)
    xr.sum(dim=(0, 2)).sum().backward()
    ref = xr.grad
    assert result.shape == x.shape, f'shape: {result.shape}'
    assert t.allclose(result, ref), 'values mismatch'
    # keepdim=True branch
    out2 = x.sum(dim=(0, 2), keepdim=True)
    grad2 = t.ones_like(out2)
    r2 = sum_back_multi(grad2, x, dims=(0, 2), keepdim=True)
    assert r2.shape == x.shape


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

def sum_back_multi(grad_out: t.Tensor, x: t.Tensor, dims: tuple, keepdim: bool = False) -> t.Tensor:
    if keepdim:
        return grad_out.expand(x.shape)
    g = grad_out
    for d in sorted(dims):
        g = g.unsqueeze(d)
    return g.expand(x.shape)

# Quick test
t.manual_seed(5)
x = t.randn(2, 3, 4)
out = x.sum(dim=(0, 2))           # shape (3,)
grad_out = t.ones_like(out)
result = sum_back_multi(grad_out, x, dims=(0, 2), keepdim=False)
print('result.shape:', result.shape)  # expect (2, 3, 4)
```
</details>